# Prework - Čištění dat (Cleaning Data: Heart Disease Dataset)

Dataset `heart_data.csv` obsahuje diagnostické lékařské parametry pacientů pro určení diagnózy srdečního onemocnění (**ahd** - Angina / Heart Disease: `yes` / `no`).

### Seznam proměnných v datasetu:
- `age`: věk pacienta
- `sex`: pohlaví pacienta (`Male` / `Female`)
- `chestpain`: typ bolesti na hrudi (`typical`, `asymptomatic`, `nonanginal`, `nontypical`)
- `trestbps` (v datasetu `restbp`): klidový krevní tlak (v mmHg)
- `chol`: hladina cholesterolu v krvi (v mg/dl)
- `fbs`: hladina cukru v krvi nalačno (> 120 mg/dl $\rightarrow$ `Yes`, jinak `No`)
- `restecg`: výsledky klidového EKG vyšetření
- `minhr`: minimální naměřená tepová frekvence
- `maxhr`: maximální naměřená tepová frekvence
- `exang`: cvičením vyvolaná angina pectoris (`Yes` / `No`)
- `oldpeak`: deprese ST segmentu vyvolaná zátěží vzhledem ke klidu
- `slope`: sklon ST segmentu
- `ca`: počet velkých cév barvených fluoroskopií (0–3)
- `thal`: thalasémie (genetické onemocnění krve: `normal`, `fixed`, `reversable`)
- `ahd`: diagnóza srdečního onemocnění (cílová proměnná: `Yes` / `No`)

## 1. a 2. Načtení dat a zobrazení posledních 15 pozorování

In [ ]:
import os
import pandas as pd

# Načtení dat
data_path = os.path.join("data", "heart_data.csv")
df = pd.read_csv(data_path)

print(f"Rozměry načtených dat: {df.shape[0]} řádků, {df.shape[1]} sloupců")
# Zobrazení posledních 15 pozorování
df.tail(15)

## 3. Odstranění redundantních proměnných

Prozkoumáme výskyt prázdných hodnot a identifikujeme sloupce, které nenesou žádnou informaci. Sloupec `minhr` obsahuje 100 % chybějících hodnot (`NaN` ve všech 313 řádcích) a je pro model zcela nepoužitelný.

In [ ]:
# Analýza chybějících hodnot ve všech sloupcích
print(df.isnull().sum())

# Odstranění sloupce 'minhr'
df = df.drop(columns=["minhr"])
print(f"\nNový tvar datasetu po odstranění 'minhr': {df.shape}")

## 4. Detekce a odstranění duplicit

Pomocí `.duplicated().sum()` zjistíme počet duplicitních záznamů a pomocí `.drop_duplicates()` je odstraníme.

In [ ]:
num_dupes = df.duplicated().sum()
print(f"Počet nalezených duplicitních řádků: {num_dupes}")

if num_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Počet unikátních řádků po odstranění duplicit: {len(df)}")

## 5. Ošetření chybějících hodnot (Imputace)

Dle zadání doplníme chybějící hodnoty v numerických proměnných jejich **mediánem** (`.median()`). V kategorickém sloupci `thal` doplníme chybějící hodnoty nejčastější hodnotou (**modus**).

In [ ]:
print("Chybějící hodnoty před imputací:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Doplnění numerických sloupců mediánem
for col in ["chol", "ca"]:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"Sloupec '{col}' doplněn mediánem: {median_value}")

# Doplnění kategorického sloupce thal modem
thal_mode = df["thal"].mode()[0]
df["thal"] = df["thal"].fillna(thal_mode)
print(f"Sloupec 'thal' doplněn modem: '{thal_mode}'")

print(f"\nCelkový počet zbývajících chybějících hodnot: {df.isnull().sum().sum()}")

## 6. Kontrola a oprava chyb v kategorických proměnných

V textových polích se často vyskytují nestejné velikosti písmen (např. `Male`/`male`, `Yes`/`yes`, `FIXED`/`fixed`) a překlepy:
- `chestpain`: překlepy jako `nontypica`, `asymptomaticc`, `nontypiccal`
- `thal`: překlep `reversabble`

In [ ]:
cat_cols = ["sex", "chestpain", "fbs", "exang", "thal", "ahd"]

# Sjednocení na malá písmena
for col in cat_cols:
    df[col] = df[col].astype(str).str.lower()

# Oprava překlepů v 'chestpain'
chestpain_fixes = {
    "nontypica": "nontypical",
    "asymptomaticc": "asymptomatic",
    "nontypiccal": "nontypical"
}
df["chestpain"] = df["chestpain"].replace(chestpain_fixes)

# Oprava překlepu v 'thal'
thal_fixes = {"reversabble": "reversable"}
df["thal"] = df["thal"].replace(thal_fixes)

# Kontrola unikátních hodnot po opravě
for col in cat_cols:
    print(f"{col:10s}: {sorted(df[col].unique())}")

## 7. Uložení vyčištěného datasetu do .csv bez indexu

Vyčištěná data uložíme jako `heart_data_exercise_1.csv` s argumentem `index=False`.

In [ ]:
output_path = os.path.join("data", "heart_data_exercise_1.csv")
df.to_csv(output_path, index=False)
print(f"Vyčištěný dataset uložen do '{output_path}'")
print(f"Finální rozměry: {df.shape[0]} řádků, {df.shape[1]} sloupců")
df.head()